# Week 3c: CPU serving of the GGUF model (llama.cpp)

**Kaggle settings:** Accelerator = *None* (CPU only, no GPU quota), Internet = *on*.

`python scripts/kaggle_run.py run --week 3 --part c --cpu --kernel-source qwen-sql-week3a`

The deployment target has no GPU (free cloud tiers), so this is the serving study that matters: the Week 3a GGUFs on `llama-server`, one flag changed at a time vs the Q4_K_M baseline (`configs/cpu_experiments.yaml`), a fresh server per load level. Also: raw prefill/decode speed per thread count (`llama-bench`), and Spider accuracy of the baseline config on all 1,034 dev questions, served live.

In [ ]:
BASE = "Qwen/Qwen3-1.7B"
OUT = "/kaggle/working/week3c"
SPIDER = "/kaggle/working/data/spider_data"
WL = f"{OUT}/workloads"
LLAMA = "/kaggle/working/llama.cpp"
SERVER = f"{LLAMA}/build/bin/llama-server"
PLAN = "configs/cpu_experiments.yaml"

import os, subprocess, json, glob, shutil
os.makedirs(OUT, exist_ok=True)
STATUS = {}

def run(step, cmd, needs=()):
    """Run a shell step, stream its output, keep a per-step log, never raise.
    A failed step marks itself so that steps depending on it are skipped, not re-failed."""
    blocked = [n for n in needs if STATUS.get(n) != "ok"]
    if blocked:
        STATUS[step] = "skipped"
        print(f"--- skip {step}: needs {blocked}")
        return False
    print(f"--- {step}: {cmd}")
    with open(f"{OUT}/{step}.log", "w") as log:
        proc = subprocess.Popen(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in proc.stdout:
            log.write(line)
            if "it/s]" not in line and "s/it]" not in line:
                print(line, end="")
    STATUS[step] = "ok" if proc.wait() == 0 else "failed"
    print(f"--- {step}: {STATUS[step]}")
    return STATUS[step] == "ok"

In [ ]:
import glob, shutil, subprocess, tarfile
DEST = "/kaggle/working/repo"
# Search only the code dataset: a mounted kernel output (e.g. week3a) contains its own repo copy,
# and an unscoped /kaggle/input/** glob picked that up instead (week3b run 1 ran week3a's code).
roots = [d for d in glob.glob("/kaggle/input/**/qwen-inference-optimization-src", recursive=True) if os.path.isdir(d)]
marker = glob.glob(f"{roots[0]}/COMMIT") if roots else []
if marker:  # code uploaded by scripts/kaggle_run.py
    commit = open(marker[0]).read().strip()
    tars = glob.glob(f"{roots[0]}/**/src.tar", recursive=True)
    trees = sorted(glob.glob(f"{roots[0]}/**/pyproject.toml", recursive=True), key=len)  # shallowest = repo root
    os.makedirs(DEST, exist_ok=True)
    if tars:
        with tarfile.open(tars[0]) as t:
            t.extractall(DEST, filter="data")
    elif trees:  # Kaggle unpacks uploaded archives itself (into src/)
        shutil.copytree(os.path.dirname(trees[0]), DEST, dirs_exist_ok=True)
    else:
        raise RuntimeError("code dataset attached but neither src.tar nor pyproject.toml found")
else:
    !git clone -q {REPO} {DEST}
    commit = subprocess.getoutput(f"git -C {DEST} rev-parse --short HEAD")
open(f"{OUT}/COMMIT", "w").write(commit + "\n")
print("code commit:", commit)
%cd {DEST}
!pip install -q -e . 2>&1 | tail -3  # CPU only: no GPU stack

# Fail fast: a broken setup must not burn GPU hours on cells that can only fail.
check = subprocess.run(["python", "-c", "import bench.cpu_experiments, bench.matrix, evaluation.remote, quantize.gguf, transformers"],
                       capture_output=True, text=True)
if check.returncode != 0:
    raise RuntimeError("repo or serving deps not importable:\n" + check.stderr[-2000:])
print("repo and serving deps import OK")

In [ ]:
info = glob.glob("/kaggle/input/**/models/gguf/gguf_info.json", recursive=True)
if not info:
    raise RuntimeError("week3a output not mounted: run with --kernel-source qwen-sql-week3a")
GGUF = os.path.dirname(info[0])
A_OUT = os.path.join(os.path.dirname(os.path.dirname(GGUF)), "week3a")  # week3a quality results (fp16 preds)
print("gguf:", sorted(os.listdir(GGUF)), "\nweek3a results:", A_OUT, os.path.isdir(A_OUT))

## 1. Machine record (CPU model, cores, ISA, memory: CPU numbers mean nothing without it)

In [ ]:
!lscpu | tee {OUT}/lscpu.txt | grep -E "Model name|^CPU\(s\)|Thread|Core|L3|MHz"
!grep -o -w -E "avx2|avx512f|avx512_vnni|avx_vnni|fma|f16c" /proc/cpuinfo | sort -u | tr "\n" " " | tee {OUT}/isa.txt
!free -g | tee {OUT}/mem.txt

## 2. Data, frozen workloads, llama.cpp (pinned tag from configs/quant.yaml)

In [ ]:
!python scripts/get_spider.py --out /kaggle/working/data
run("workloads", f"python -m bench.workloads --tokenizer {BASE} --out {WL} --spider-dir {SPIDER}")
run("llama_build", f"python -m quantize.gguf --setup-only --llama-cpp {LLAMA}")

## 3. Raw engine speed: prefill (pp512) and decode (tg128) tokens/s by thread count
Prefill is matrix-matrix work (compute-bound: should scale with threads); decode is matrix-vector (memory-bandwidth-bound: should flatten early).

In [ ]:
q4, q8 = (f"{GGUF}/" + json.load(open(info[0]))["files"][t]["file"] for t in ("Q4_K_M", "Q8_0"))
run("llama_bench", f"{LLAMA}/build/bin/llama-bench -m {q4},{q8} -t 1,2,4 -p 512 -n 128 -r 2 -o md "
    f"| tee {OUT}/llama_bench.md", needs=["llama_build"])

## 4. Serving experiments (configs/cpu_experiments.yaml)

In [ ]:
run("cpu_experiments", f"python -m bench.cpu_experiments --plan {PLAN} --llama-server {SERVER} --gguf-dir {GGUF} "
    f"--workloads {WL} --out {OUT}/experiments", needs=["workloads", "llama_build"])

## 5. Accuracy of the deployed config: all 1,034 dev questions through llama-server (4 slots)
Week 3a measured 200 questions twice and got 52.5% and 56.0%: the full set removes most of that sampling noise.

In [ ]:
if run("quality_q4_k_m", f"python -m bench.cpu_experiments --plan {PLAN} --llama-server {SERVER} --gguf-dir {GGUF} "
       f"--quality q4_k_m --tokenizer {BASE} --spider-dir {SPIDER} --out {OUT}/preds", needs=["llama_build"]):
    run("compare_q4_k_m", f"python -m evaluation.compare --spider-dir {SPIDER} --a {A_OUT}/preds/fp16.jsonl "
        f"--b {OUT}/preds/llamacpp_q4_k_m.jsonl --out {OUT}/compare/llamacpp_q4_k_m.json")

## 6. Matrix + Pareto, results zip

In [ ]:
run("matrix", f"python -m bench.matrix --plan {PLAN} --experiments {OUT}/experiments --quality {A_OUT} {OUT} --out {OUT}/report")
shutil.copy(f"{WL}/MANIFEST.json", OUT) if os.path.exists(f"{WL}/MANIFEST.json") else print("missing MANIFEST")
shutil.rmtree(WL, ignore_errors=True)
json.dump(STATUS, open(f"{OUT}/status.json", "w"), indent=2)
print("step status:", STATUS)
shutil.make_archive("/kaggle/working/week3c_results", "zip", OUT)
for d in ["/kaggle/working/data", LLAMA]:
    shutil.rmtree(d, ignore_errors=True)
print("download /kaggle/working/week3c_results.zip")